# **NOTEBOOK 01: DATA PREPROCESSING PIPELINE**

**MedMind: From Risk to Action — An Explainable ML Screening Tool for Mental Health in Spanish Medical Students**
María Reiter Hernández · IE University · Capstone Project 2026

---

**Purpose:** Clean the raw DABE 2020 dataset and construct the analytical datasets used by Notebooks 02 and 03.

**Inputs:** Raw DABE 2020 survey file (Capdevila-Gaudens et al., 2021).

**Outputs:**
- `dabe_clean_full.csv` — all variables plus computed scale scores, used for exploratory data analysis in Notebook 02 and by the institutional dashboard of the deployed Streamlit application
- `dabe_ml_features.csv` — feature matrix and binary targets, used for machine learning modelling in Notebook 03
- `preprocessing_report.txt` — summary statistics and validation output for verification against the original DABE publication

**Key operations:** removal of 7 empty rows and 7 null separator columns; age capping (17–50); gender/orientation/employment recoding; MBI-SS item-to-subscale remapping validated against the original DABE prevalence figures; STAI and Jefferson reverse-scoring; perceived-problems parser handling three encoding formats; psychopharmaceutical type parsing; construction of the four binary targets (BDI-II ≥ 20 for depression; MBI-SS Exhaustion ≥ 2.8 AND Cynicism ≥ 2.25 for burnout; STAI-T ≥ 56 for anxiety; BDI Item 9 > 0 for suicidal ideation).

**Reproduces:** Section 3.5 (Data Preprocessing) of the thesis.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

In [ ]:
INPUT_FILE = '/content/Datos Proyecto DABE_SEDEM-CEEM english.xlsx'

In [ ]:
# MBI-SS subscale definitions (verified against Capdevila-Gaudens 2021)
# These are SURVEY ITEM NUMBERS, not standard MBI-SS item numbers.
# Validated: this grouping reproduces the published 36.7% burnout prevalence.
MBI_EXHAUSTION_ITEMS = ['M1', 'M5', 'M8', 'M11', 'M14']
MBI_CYNICISM_ITEMS   = ['M2', 'M3', 'M9', 'M12']
MBI_EFFICACY_ITEMS   = ['M4', 'M6', 'M7', 'M10', 'M13', 'M15']

In [ ]:
# Burnout thresholds from Galán et al. (2011), as used in the DABE paper
EXHAUSTION_HIGH_THRESHOLD = 2.8
CYNICISM_HIGH_THRESHOLD   = 2.25
EFFICACY_LOW_THRESHOLD    = 3.84

In [ ]:
# STAI reverse-scored items (standard STAI-Y form)
# These items are "anxiety-absent" — higher raw scores mean LESS anxiety
# Formula: reversed = 5 - raw_value
STAI_STATE_REVERSE = [1, 2, 5, 8, 10, 11, 15, 16, 19, 20]  # AHO items
STAI_TRAIT_REVERSE = [1, 6, 7, 10, 13, 16, 19]               # HAB items

In [ ]:
# Jefferson Empathy Scale reverse-scored items (S-version for students)
# Formula: reversed = 8 - raw_value
JEFFERSON_REVERSE = [1, 3, 6, 7, 8, 11, 12, 14, 18, 19]

In [ ]:
# BDI-II depression categories (Beck et al. 1996)
BDI_CATEGORIES = {
    'None/Minimal': (0, 13),
    'Mild': (14, 19),
    'Moderate': (20, 28),
    'Severe': (29, 63),
}

In [ ]:
# STAI anxiety categories (as used in DABE paper)
STAI_CATEGORIES = {
    'Very Low': (20, 31),
    'Low': (32, 43),
    'Moderate': (44, 55),
    'High': (56, 67),
    'Very High': (68, 80),
}

In [ ]:
# Age bounds for outlier handling
AGE_MIN = 17
AGE_MAX = 50

In [ ]:
# Perception of problems category mapping
PROBLEM_CATEGORIES = {
    1: 'prob_academic_performance',
    2: 'prob_family',
    3: 'prob_daily_tasks',
    4: 'prob_partner',
    5: 'prob_time_management',
    6: 'prob_health',
    7: 'prob_money',
    8: 'prob_alcohol',
    9: 'prob_colleagues',
    10: 'prob_tobacco',
    11: 'prob_teachers',
    12: 'prob_other_drugs',
}

In [ ]:
# Psychopharmaceutical categories
PHARMA_CATEGORIES = {
    1: 'pharma_anxiolytics',
    2: 'pharma_mood_stabilizers',
    3: 'pharma_antidepressants',
    4: 'pharma_antipsychotics',
    5: 'pharma_other',
}

### **1. Load Data**

In [ ]:
print("\n" + "=" * 70)
print("STEP 1: Loading raw data")
print("=" * 70)

df = pd.read_excel(INPUT_FILE, sheet_name='Answers')
print(f"  Raw shape: {df.shape[0]} rows × {df.shape[1]} columns")


STEP 1: Loading raw data
  Raw shape: 5223 rows × 129 columns


### **2. Drop Junk Columns**

In [ ]:
print("\n" + "=" * 70)
print("STEP 2: Dropping empty separator columns")
print("=" * 70)

unnamed_cols = [c for c in df.columns if 'Unnamed' in str(c)]
print(f"  Dropping {len(unnamed_cols)} unnamed columns: {unnamed_cols}")
df = df.drop(columns=unnamed_cols)
print(f"  Shape after: {df.shape[0]} rows × {df.shape[1]} columns")


STEP 2: Dropping empty separator columns
  Dropping 7 unnamed columns: ['Unnamed: 23', 'Unnamed: 45', 'Unnamed: 66', 'Unnamed: 87', 'Unnamed: 108', 'Unnamed: 127', 'Unnamed: 128']
  Shape after: 5223 rows × 122 columns


### **3. Drop Completely Empty Rows**

In [ ]:
print("\n" + "=" * 70)
print("STEP 3: Dropping rows with no data")
print("=" * 70)

# These 7 rows have NaN for University, Course, Gender, and all scale items
core_cols = ['University', 'Course', 'Gender']
empty_mask = df[core_cols].isnull().all(axis=1)
n_empty = empty_mask.sum()
print(f"  Found {n_empty} rows with missing University/Course/Gender")
df = df[~empty_mask].reset_index(drop=True)
print(f"  Shape after: {df.shape[0]} rows × {df.shape[1]} columns")


STEP 3: Dropping rows with no data
  Found 7 rows with missing University/Course/Gender
  Shape after: 5216 rows × 122 columns


### **4. Clean Age Outliers**

In [ ]:
print("\n" + "=" * 70)
print("STEP 4: Handling age outliers")
print("=" * 70)

n_below = (df['Age'] < AGE_MIN).sum()
n_above = (df['Age'] > AGE_MAX).sum()
n_missing = df['Age'].isnull().sum()
print(f"  Ages below {AGE_MIN}: {n_below}")
print(f"  Ages above {AGE_MAX}: {n_above}")
print(f"  Ages missing: {n_missing}")

# Cap rather than drop — preserves sample size
df['Age'] = df['Age'].clip(lower=AGE_MIN, upper=AGE_MAX)
# Fill the 2 missing ages with median
df['Age'] = df['Age'].fillna(df['Age'].median())
print(f"  After clipping [{AGE_MIN}-{AGE_MAX}]: mean={df['Age'].mean():.1f}, "
      f"median={df['Age'].median():.0f}, min={df['Age'].min():.0f}, max={df['Age'].max():.0f}")


STEP 4: Handling age outliers
  Ages below 17: 1
  Ages above 50: 4
  Ages missing: 2
  After clipping [17-50]: mean=21.4, median=21, min=17, max=50


### **5. Recode Categorical Variables**

In [ ]:
print("\n" + "=" * 70)
print("STEP 5: Recoding categorical variables")
print("=" * 70)

# --- Course: extract integer from "1º", "2º", etc. ---
df['course_year'] = df['Course'].str.extract(r'(\d)').astype(float).astype('Int64')
print(f"  Course year distribution:\n{df['course_year'].value_counts().sort_index().to_string()}")

# --- Gender: 1=Male, 2=Female, 3/4=Other (merged, n=42) ---
df['gender'] = df['Gender'].map({1.0: 'Male', 2.0: 'Female', 3.0: 'Other', 4.0: 'Other'})
# Binary for modeling
df['gender_female'] = (df['Gender'] == 2.0).astype(int)
print(f"  Gender: {df['gender'].value_counts().to_dict()}")

# --- Sexual orientation ---
df['orientation'] = df['Sexual orientation'].map({
    1.0: 'Heterosexual', 2.0: 'Homosexual', 3.0: 'Bisexual', 4.0: 'Prefer not to say'
})
# Binary for modeling (non-heterosexual may have different risk profile)
df['orientation_non_hetero'] = (df['Sexual orientation'].isin([2.0, 3.0])).astype(int)
print(f"  Orientation: {df['orientation'].value_counts().to_dict()}")

# --- Employment ---
df['works'] = df['Do you also work?'].map({1.0: 'No', 2.0: 'Full-time', 3.0: 'Part-time'})
df['works_any'] = (df['Do you also work?'].isin([2.0, 3.0])).astype(int)
print(f"  Works: {df['works'].value_counts().to_dict()}")

# --- Fellowship/Scholarship ---
df['has_fellowship'] = (df['Do you have fellowship?'] == 2.0).astype(int)
print(f"  Fellowship: {df['has_fellowship'].value_counts().to_dict()}")

# --- Attendance (already ordinal 1-6, keep as is) ---
df['attendance'] = df['Attendance to non-compulsory classroom teaching activities']

# --- Life Events: recode 1=Yes/2=No to 1/0 ---
for i in range(1, 6):
    col = f'EVE{i}'
    df[f'eve_{i}'] = (df[col] == 1.0).astype(int)

eve_labels = {
    1: 'Serious illness/accident',
    2: 'Family member serious illness',
    3: 'Serious financial problems',
    4: 'Relationship breakup',
    5: 'Death of close person'
}
for i, label in eve_labels.items():
    n = df[f'eve_{i}'].sum()
    print(f"  EVE{i} ({label}): {n} ({n/len(df)*100:.1f}%)")

# --- Social Support ---
# APOYO1: number of close people (1=None, 2=1-2, 3=3-5, 4=5+) — keep ordinal
df['social_support_n'] = df['APOYO1']
# APOYO2-4: quality (1=A lot ... 5=None) — REVERSE so higher=better
for i in range(2, 5):
    df[f'social_support_q{i}'] = 6 - df[f'APOYO{i}']  # Now 1=None, 5=A lot

# --- Academic Performance ---
# ACAD1: effort-grade match (1=Lower, 2=Concordant, 3=Higher) — keep as is
df['acad_effort_match'] = df['ACAD1']
# ACAD2-3: satisfaction (1=Very satisfied ... 4=Not at all) — REVERSE so higher=better
df['acad_satisfaction_studies'] = 5 - df['ACAD2']
df['acad_satisfaction_choice'] = 5 - df['ACAD3']

# --- Substance Use ---
# Tobacco: 1=Yes, 2=No, 3=Occasional → recode to ordinal 0/1/2
df['tobacco'] = df['Tobacco smoker'].map({2.0: 0, 3.0: 1, 1.0: 2})  # 0=No, 1=Occasional, 2=Regular
# Cannabis: already ordinal 1-4, recode to 0-3
df['cannabis'] = df['Cannabis smoker'] - 1  # 0=Never, 1=Tried, 2=Occasional, 3=Regular
# Alcohol: already ordinal 1-5, recode to 0-4
df['alcohol'] = df['Alcohol'] - 1  # 0=Never ... 4=3+ times/week
# Psychopharmaceuticals: 1=Yes, 2=No → binary
df['takes_psychopharm'] = (df['Psychopharmaceuticals'] == 1.0).astype(int)

print(f"  Tobacco (0=No, 1=Occ, 2=Reg): {df['tobacco'].value_counts().sort_index().to_dict()}")
print(f"  Cannabis (0-3): {df['cannabis'].value_counts().sort_index().to_dict()}")
print(f"  Alcohol (0-4): {df['alcohol'].value_counts().sort_index().to_dict()}")
print(f"  Psychopharmaceuticals: {df['takes_psychopharm'].sum()} ({df['takes_psychopharm'].mean()*100:.1f}%)")


STEP 5: Recoding categorical variables
  Course year distribution:
course_year
1    1063
2     985
3     929
4     841
5     885
6     513
  Gender: {'Female': 3979, 'Male': 1195, 'Other': 42}
  Orientation: {'Heterosexual': 4029, 'Bisexual': 737, 'Homosexual': 254, 'Prefer not to say': 196}
  Works: {'No': 4428, 'Part-time': 727, 'Full-time': 61}
  Fellowship: {0: 3511, 1: 1705}
  EVE1 (Serious illness/accident): 846 (16.2%)
  EVE2 (Family member serious illness): 896 (17.2%)
  EVE3 (Serious financial problems): 1131 (21.7%)
  EVE4 (Relationship breakup): 506 (9.7%)
  EVE5 (Death of close person): 1309 (25.1%)
  Tobacco (0=No, 1=Occ, 2=Reg): {0: 4392, 1: 417, 2: 407}
  Cannabis (0-3): {0.0: 3135, 1.0: 1782, 2.0: 227, 3.0: 68}
  Alcohol (0-4): {0.0: 829, 1.0: 1762, 2.0: 1984, 3.0: 526, 4.0: 115}
  Psychopharmaceuticals: 1063 (20.4%)


### **6. Compute Composite Psychological Scores**

In [ ]:
print("\n" + "=" * 70)
print("STEP 6: Computing composite psychological scores")
print("=" * 70)

# --- 6a: BDI-II Depression Total ---
bdi_items = [f'B{i}' for i in range(1, 22)]
df['bdi_total'] = df[bdi_items].sum(axis=1)

# Categorize
def categorize_bdi(score):
    if pd.isna(score):
        return np.nan
    for cat, (low, high) in BDI_CATEGORIES.items():
        if low <= score <= high:
            return cat
    return np.nan

df['depression_category'] = df['bdi_total'].apply(categorize_bdi)
print(f"\n  BDI-II Total: mean={df['bdi_total'].mean():.1f}, std={df['bdi_total'].std():.1f}")
print(f"  Depression categories:\n{df['depression_category'].value_counts().to_string()}")
print(f"  Any depression (BDI>=14): {(df['bdi_total'] >= 14).sum()} "
      f"({(df['bdi_total'] >= 14).mean()*100:.1f}%) — [Published: 41%]")

# --- 6b: MBI-SS Burnout Subscales ---
# Using codification sheet grouping (validated against published paper)
df['mbi_exhaustion'] = df[MBI_EXHAUSTION_ITEMS].mean(axis=1)
df['mbi_cynicism'] = df[MBI_CYNICISM_ITEMS].mean(axis=1)
df['mbi_efficacy'] = df[MBI_EFFICACY_ITEMS].mean(axis=1)

print(f"\n  MBI-SS Exhaustion: mean={df['mbi_exhaustion'].mean():.2f}, std={df['mbi_exhaustion'].std():.2f}")
print(f"  MBI-SS Cynicism:   mean={df['mbi_cynicism'].mean():.2f}, std={df['mbi_cynicism'].std():.2f}")
print(f"  MBI-SS Efficacy:   mean={df['mbi_efficacy'].mean():.2f}, std={df['mbi_efficacy'].std():.2f}")

burnout_mask = (df['mbi_exhaustion'] >= EXHAUSTION_HIGH_THRESHOLD) & \
               (df['mbi_cynicism'] >= CYNICISM_HIGH_THRESHOLD)
print(f"  Burnout (EX>={EXHAUSTION_HIGH_THRESHOLD} & CY>={CYNICISM_HIGH_THRESHOLD}): "
      f"{burnout_mask.sum()} ({burnout_mask.mean()*100:.1f}%) — [Published: ~37%]")

# --- 6c: STAI State Anxiety (AHO1-AHO20) ---
stai_s_items = [f'AHO{i}' for i in range(1, 21)]

# Reverse-score anxiety-absent items
df_stai_s = df[stai_s_items].copy()
for item_num in STAI_STATE_REVERSE:
    col = f'AHO{item_num}'
    df_stai_s[col] = 5 - df_stai_s[col]

df['stai_state_total'] = df_stai_s.sum(axis=1)

# Categorize
def categorize_stai(score):
    if pd.isna(score):
        return np.nan
    for cat, (low, high) in STAI_CATEGORIES.items():
        if low <= score <= high:
            return cat
    return np.nan

df['anxiety_state_category'] = df['stai_state_total'].apply(categorize_stai)
print(f"\n  STAI State Total: mean={df['stai_state_total'].mean():.1f}, "
      f"std={df['stai_state_total'].std():.1f}")
print(f"  State anxiety categories:\n{df['anxiety_state_category'].value_counts().to_string()}")

# --- 6d: STAI Trait Anxiety (HAB1-HAB20) ---
stai_t_items = [f'HAB{i}' for i in range(1, 21)]

df_stai_t = df[stai_t_items].copy()
for item_num in STAI_TRAIT_REVERSE:
    col = f'HAB{item_num}'
    df_stai_t[col] = 5 - df_stai_t[col]

df['stai_trait_total'] = df_stai_t.sum(axis=1)
df['anxiety_trait_category'] = df['stai_trait_total'].apply(categorize_stai)
print(f"\n  STAI Trait Total: mean={df['stai_trait_total'].mean():.1f}, "
      f"std={df['stai_trait_total'].std():.1f}")
print(f"  Trait anxiety categories:\n{df['anxiety_trait_category'].value_counts().to_string()}")

# --- 6e: Jefferson Empathy Scale ---
jeff_items = [f'E{i}' for i in range(1, 21)]

df_jeff = df[jeff_items].copy()
for item_num in JEFFERSON_REVERSE:
    col = f'E{item_num}'
    df_jeff[col] = 8 - df_jeff[col]

df['empathy_total'] = df_jeff.sum(axis=1)
print(f"\n  Jefferson Empathy: mean={df['empathy_total'].mean():.1f}, "
      f"std={df['empathy_total'].std():.1f}, range=[{df['empathy_total'].min():.0f}-{df['empathy_total'].max():.0f}]")


STEP 6: Computing composite psychological scores

  BDI-II Total: mean=13.5, std=10.6
  Depression categories:
depression_category
None/Minimal    3075
Mild             919
Moderate         689
Severe           533
  Any depression (BDI>=14): 2141 (41.0%) — [Published: 41%]

  MBI-SS Exhaustion: mean=2.46, std=1.10
  MBI-SS Cynicism:   mean=3.46, std=0.90
  MBI-SS Efficacy:   mean=2.70, std=0.73
  Burnout (EX>=2.8 & CY>=2.25): 1918 (36.8%) — [Published: ~37%]

  STAI State Total: mean=45.8, std=13.8
  State anxiety categories:
anxiety_state_category
Low          1576
Moderate     1423
High          984
Very Low      874
Very High     359

  STAI Trait Total: mean=46.5, std=11.2
  Trait anxiety categories:
anxiety_trait_category
Moderate     1800
Low          1748
High          991
Very Low      477
Very High     200

  Jefferson Empathy: mean=120.0, std=12.5, range=[32-140]


### **7. Parse Multi-select Columns**

In [ ]:
print("\n" + "=" * 70)
print("STEP 7: Parsing multi-select columns")
print("=" * 70)

# --- 7a: Perception of Problems or Difficulties ---
# This column has three formats:
#   - Single integer (e.g., 6) → one category
#   - Decimal (e.g., 1.5 → categories 1 and 5; 4.11 → categories 4 and 11)
#   - Comma-separated string (e.g., "2, 3, 5, 6, 9, 11")

def parse_problems(value):
    """Parse a Perception of Problems value into a set of category integers."""
    if pd.isna(value):
        return set()

    # If it's a string with commas → split on commas
    if isinstance(value, str) and ',' in str(value):
        try:
            return set(int(x.strip()) for x in str(value).split(',') if x.strip())
        except ValueError:
            return set()

    # If it's numeric
    try:
        num = float(value)
    except (ValueError, TypeError):
        return set()

    # Check if it's a whole number (single category)
    if num == int(num):
        return {int(num)}

    # It's a decimal representing two categories
    # Split on the decimal point: "1.5" → "1" and "5", "4.11" → "4" and "11"
    str_val = str(value)
    if '.' in str_val:
        parts = str_val.split('.')
        try:
            cat1 = int(parts[0])
            cat2 = int(parts[1])
            # Validate both are in range 1-12
            result = set()
            if 1 <= cat1 <= 12:
                result.add(cat1)
            if 1 <= cat2 <= 12:
                result.add(cat2)
            return result
        except ValueError:
            return set()

    return set()

# Initialize all problem dummies as 0
for cat_num, col_name in PROBLEM_CATEGORIES.items():
    df[col_name] = 0

# Parse each row
prob_col = df['Perception of problems or difficulties']
for idx in df.index:
    categories = parse_problems(prob_col[idx])
    for cat_num in categories:
        if cat_num in PROBLEM_CATEGORIES:
            df.at[idx, PROBLEM_CATEGORIES[cat_num]] = 1

# Report
print("\n  Problem prevalence:")
for cat_num, col_name in PROBLEM_CATEGORIES.items():
    n = df[col_name].sum()
    print(f"    {cat_num:2d}. {col_name.replace('prob_', ''):30s}: {n:4d} ({n/len(df)*100:.1f}%)")

# Count total problems per student
df['n_problems'] = sum(df[col] for col in PROBLEM_CATEGORIES.values())
print(f"\n  Problems per student: mean={df['n_problems'].mean():.1f}, max={df['n_problems'].max()}")

# --- 7b: Type of Psychopharmaceuticals ---
# Same parsing logic, but only for students taking psychopharmaceuticals

for cat_num, col_name in PHARMA_CATEGORIES.items():
    df[col_name] = 0

pharma_col = df['What psychopharmaceuticals']
for idx in df.index:
    value = pharma_col[idx]
    if pd.isna(value):
        continue
    categories = parse_problems(value)  # Same parsing logic works
    for cat_num in categories:
        if cat_num in PHARMA_CATEGORIES:
            df.at[idx, PHARMA_CATEGORIES[cat_num]] = 1

print("\n  Psychopharmaceutical types (among users):")
n_users = df['takes_psychopharm'].sum()
for cat_num, col_name in PHARMA_CATEGORIES.items():
    n = df[col_name].sum()
    print(f"    {col_name.replace('pharma_', ''):25s}: {n:4d} ({n/n_users*100:.1f}% of users)")


STEP 7: Parsing multi-select columns

  Problem prevalence:
     1. academic_performance          : 3250 (62.3%)
     2. family                        : 1596 (30.6%)
     3. daily_tasks                   : 2535 (48.6%)
     4. partner                       : 1207 (23.1%)
     5. time_management               : 3938 (75.5%)
     6. health                        : 2935 (56.3%)
     7. money                         : 1759 (33.7%)
     8. alcohol                       :  242 (4.6%)
     9. colleagues                    : 1533 (29.4%)
    10. tobacco                       :  361 (6.9%)
    11. teachers                      :  304 (5.8%)
    12. other_drugs                   :  100 (1.9%)

  Problems per student: mean=3.8, max=12

  Psychopharmaceutical types (among users):
    anxiolytics              :  851 (80.1% of users)
    mood_stabilizers         :   28 (2.6% of users)
    antidepressants          :  532 (50.0% of users)
    antipsychotics           :   57 (5.4% of users)
    other 

### **8. Creating Target Variables**

In [ ]:
print("\n" + "=" * 70)
print("STEP 8: Creating ML target variables")
print("=" * 70)

# Target 1: Clinical depression (BDI >= 20: moderate or severe)
df['target_depression_clinical'] = (df['bdi_total'] >= 20).astype(int)

# Target 2: Burnout (high exhaustion AND high cynicism)
df['target_burnout'] = ((df['mbi_exhaustion'] >= EXHAUSTION_HIGH_THRESHOLD) &
                        (df['mbi_cynicism'] >= CYNICISM_HIGH_THRESHOLD)).astype(int)

# Target 3: High trait anxiety (>= 75th percentile of sample)
STAI_HIGH_THRESHOLD = 56
df['target_anxiety_high'] = (df['stai_trait_total'] >= STAI_HIGH_THRESHOLD).astype(int)

# Target 4: Suicidal ideation (B9 > 0)
df['target_suicidal_ideation'] = (df['B9'] > 0).astype(int)

print(f" Target 1 – Clinical Depression (BDI>=20): "
      f"{df['target_depression_clinical'].sum()} "
      f"({df['target_depression_clinical'].mean()*100:.1f}%)")

print(f" Target 2 – Burnout (Ex>={EXHAUSTION_HIGH_THRESHOLD} & "
      f"Cy>={CYNICISM_HIGH_THRESHOLD}): "
      f"{df['target_burnout'].sum()} "
      f"({df['target_burnout'].mean()*100:.1f}%)")

print(f" Target 3 – High Trait Anxiety (>= STAI {STAI_HIGH_THRESHOLD}): "
      f"{df['target_anxiety_high'].sum()} "
      f"({df['target_anxiety_high'].mean()*100:.1f}%)")

print(f" Target 4 – Suicidal Ideation (B9>0): "
      f"{df['target_suicidal_ideation'].sum()} "
      f"({df['target_suicidal_ideation'].mean()*100:.1f}%)")


STEP 8: Creating ML target variables
 Target 1 – Clinical Depression (BDI>=20): 1222 (23.4%)
 Target 2 – Burnout (Ex>=2.8 & Cy>=2.25): 1918 (36.8%)
 Target 3 – High Trait Anxiety (>= STAI 56): 1191 (22.8%)
 Target 4 – Suicidal Ideation (B9>0): 551 (10.6%)


### **9. Handling Remaining Missing Values**

In [ ]:
print("\n" + "=" * 70)
print("STEP 9: Handling remaining missing values")
print("=" * 70)

# Define the feature columns we'll use for ML
feature_cols = [
    # Demographics
    'Age', 'gender_female', 'course_year', 'orientation_non_hetero',
    # Academic
    'attendance', 'has_fellowship', 'works_any',
    'acad_effort_match', 'acad_satisfaction_studies', 'acad_satisfaction_choice',
    # Social support
    'social_support_n', 'social_support_q2', 'social_support_q3', 'social_support_q4',
    # Life events
    'eve_1', 'eve_2', 'eve_3', 'eve_4', 'eve_5',
    # Perceived problems
    'prob_academic_performance', 'prob_family', 'prob_daily_tasks', 'prob_partner',
    'prob_time_management', 'prob_health', 'prob_money', 'prob_alcohol',
    'prob_colleagues', 'prob_tobacco', 'prob_teachers', 'prob_other_drugs',
    'n_problems',
    # Substance use
    'tobacco', 'cannabis', 'alcohol', 'takes_psychopharm',
    # Psychopharmaceutical types (structurally missing for non-users)
    'pharma_anxiolytics', 'pharma_mood_stabilizers', 'pharma_antidepressants',
    'pharma_antipsychotics', 'pharma_other',
    # Cross-scale features
    'mbi_exhaustion', 'mbi_cynicism', 'mbi_efficacy',
    'empathy_total', 'stai_state_total',
]

target_cols = [
    'target_depression_clinical',
    'target_burnout',
    'target_anxiety_high',
    'target_suicidal_ideation',
]

# Check missingness in feature columns
print("\n  Missing values in feature columns:")
for col in feature_cols:
    n_miss = df[col].isnull().sum()
    if n_miss > 0:
        pct = n_miss / len(df) * 100
        print(f"    {col}: {n_miss} ({pct:.2f}%)")

# Fill missing values: median for numeric, mode for categorical
for col in feature_cols:
    n_miss = df[col].isnull().sum()
    if n_miss > 0:
        if df[col].dtype in ['float64', 'int64', 'Int64', 'float32']:
            fill_val = df[col].median()
            df[col] = df[col].fillna(fill_val)
        else:
            fill_val = df[col].mode()[0]
            df[col] = df[col].fillna(fill_val)

# Verify no missing values remain in features or targets
assert df[feature_cols].isnull().sum().sum() == 0, "Still have missing features!"
assert df[target_cols].isnull().sum().sum() == 0, "Still have missing targets!"
print("\n  ✓ All missing values handled. Zero NaN in features and targets.")


STEP 9: Handling remaining missing values

  Missing values in feature columns:
    cannabis: 4 (0.08%)
    mbi_efficacy: 1 (0.02%)

  ✓ All missing values handled. Zero NaN in features and targets.


### **10. Assemble and Save Outputs**

In [ ]:
print("\n" + "=" * 70)
print("STEP 10: Saving output files")
print("=" * 70)

# Define output paths
OUTPUT_DIR = '/content/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CLEAN_FULL_PATH = os.path.join(OUTPUT_DIR, 'dabe_clean_full.csv')
ML_FEATURES_PATH = os.path.join(OUTPUT_DIR, 'dabe_ml_features.csv')

# --- Output 1: Full clean dataset (for EDA) ---
df.to_csv(CLEAN_FULL_PATH, index=False)
print(f"  ✓ Saved {CLEAN_FULL_PATH}: {df.shape[0]} rows × {df.shape[1]} columns")

# --- Output 2: ML feature matrix (for modeling) ---
# feature_cols is already defined from Step 9

target_cols = ["target_depression_clinical", "target_burnout",
               "target_anxiety_high", "target_suicidal_ideation"]

# Check which feature columns are actually present in df
present_feature_cols = [col for col in feature_cols if col in df.columns]
missing_feature_cols = [col for col in feature_cols if col not in df.columns]

if missing_feature_cols:
    print(f"\n  WARNING: The following feature columns are missing from the DataFrame and will be skipped: {missing_feature_cols}")
    print(f"  This usually indicates that a previous preprocessing step (e.g., Step 7: Parsing multi-select columns) was not executed or did not run successfully. Consider re-running all cells from the beginning.")

X_ml = df[present_feature_cols].copy()
y_ml = df[target_cols].copy()

ml_dataset = pd.concat([X_ml, y_ml], axis=1)
ml_dataset.to_csv(ML_FEATURES_PATH, index=False)
print(f"  ✓ Saved {ML_FEATURES_PATH}: {ml_dataset.shape[0]} rows × {ml_dataset.shape[1]} columns")
print(f"    Features: {len(present_feature_cols)} columns (out of {len(feature_cols)} intended)")
print(f"    Targets: {len(target_cols)} columns")

# --- Output 3: Preprocessing report ---
report_lines = []
report_lines.append("=" * 70)
report_lines.append("PREPROCESSING REPORT — DABE 2020 Dataset")
report_lines.append("=" * 70)
report_lines.append(f"\nOriginal: 5,223 rows × 129 columns")
report_lines.append(f"After cleaning: {len(df)} rows × {len(present_feature_cols)} features + {len(target_cols)} targets")
report_lines.append(f"\n--- Scale Verification (vs. Published Paper) ---")
report_lines.append(f"Depression prevalence (BDI>=14): {(df['bdi_total']>=14).mean()*100:.1f}% [Published: 41%]")
report_lines.append(f"Mod-severe depression (BDI>=20): {df['target_depression_clinical'].mean()*100:.1f}% [Published: 23.4%]")
report_lines.append(f"Burnout prevalence: {df['target_burnout'].mean()*100:.1f}% [Published: ~37%]")
report_lines.append(f"Suicidal ideation (B9>0): {df['target_suicidal_ideation'].mean()*100:.1f}% [Published: ~10%]")
report_lines.append(f"\n--- MBI-SS Subscales (Codification Sheet Grouping) ---")
report_lines.append(f"Exhaustion (M1,M5,M8,M11,M14): mean={df['mbi_exhaustion'].mean():.2f}, std={df['mbi_exhaustion'].std():.2f}")
report_lines.append(f"Cynicism (M2,M3,M9,M12):       mean={df['mbi_cynicism'].mean():.2f}, std={df['mbi_cynicism'].std():.2f}")
report_lines.append(f"Efficacy (M4,M6,M7,M10,M13,M15): mean={df['mbi_efficacy'].mean():.2f}, std={df['mbi_efficacy'].std():.2f}")
report_lines.append(f"\n--- Target Variable Class Balance ---")
for t in target_cols:
    pos = df[t].sum()
    neg = len(df) - pos
    report_lines.append(f"{t}: {pos} positive ({pos/len(df)*100:.1f}%) / {neg} negative ({neg/len(df)*100:.1f}%)")
report_lines.append(f"\n--- Feature List ({len(present_feature_cols)} total) ---")
for i, col in enumerate(present_feature_cols):
    report_lines.append(f"  {i+1:2d}. {col}")

report_text = '\n'.join(report_lines)
with open(os.path.join(OUTPUT_DIR, "preprocessing_report.txt"), "w") as f:
    f.write(report_text)
print(f"  ✓ Saved preprocessing_report.txt")

# --- Final verification ---
print("\n" + "=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)
print("\nTarget prevalences (should match paper):")
for col in target_cols:
    prev = df[col].mean()
    print(f"  {col}: {prev:.1%}")


STEP 10: Saving output files
  ✓ Saved /content/data/dabe_clean_full.csv: 5216 rows × 179 columns
  ✓ Saved /content/data/dabe_ml_features.csv: 5216 rows × 50 columns
    Features: 46 columns
    Targets: 4 columns
  ✓ Saved preprocessing_report.txt

PREPROCESSING COMPLETE

Target prevalences (should match paper):
  target_depression_clinical: 23.4%
  target_burnout: 36.8%
  target_anxiety_high: 22.8%
  target_suicidal_ideation: 10.6%


### **Final Summary**

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)
print(f"""
  Dataset: {len(df)} students from {df['University'].nunique()} universities
  Features: {len(feature_cols)} predictors
  Targets: 4 binary classification tasks

  Verification against Capdevila-Gaudens et al. (2021):
    Depression (BDI>=14):     {(df['bdi_total']>=14).mean()*100:.1f}% ← Published: 41%   ✓
    Moderate-Severe (BDI>=20): {df['target_depression_clinical'].mean()*100:.1f}% ← Published: 23.4% ✓
    Burnout:                   {df['target_burnout'].mean()*100:.1f}% ← Published: ~37%  ✓
    Suicidal Ideation:         {df['target_suicidal_ideation'].mean()*100:.1f}% ← Published: ~10%  ✓

  Output files in {OUTPUT_DIR}/:
    dabe_clean_full.csv      → For EDA (Notebook 02)
    dabe_ml_features.csv     → For ML modeling (Notebooks 03-05)
    preprocessing_report.txt → For thesis methodology chapter
""")


PREPROCESSING COMPLETE

  Dataset: 5216 students from 43 universities
  Features: 46 predictors
  Targets: 4 binary classification tasks

  Verification against Capdevila-Gaudens et al. (2021):
    Depression (BDI>=14):     41.0% ← Published: 41%   ✓
    Moderate-Severe (BDI>=20): 23.4% ← Published: 23.4% ✓
    Burnout:                   36.8% ← Published: ~37%  ✓
    Suicidal Ideation:         10.6% ← Published: ~10%  ✓

  Output files in /content/data/:
    dabe_clean_full.csv      → For EDA (Notebook 02)
    dabe_ml_features.csv     → For ML modeling (Notebooks 03-05)
    preprocessing_report.txt → For thesis methodology chapter

